# Phase 1: Data Cleaning & Joining

This notebook focuses on data cleaning and merging operations for the 6 raw CSV files in the **EcoShield AI** project workspace, grouped into 3 distinct pillars:
1. **Pillar 1: Microplastic Monitoring** (`dataset1_microplastic_sample` + `dataset1_sampling_location` -> `cleaned_microplastics.csv`)
2. **Pillar 2: Plastic Waste Cleanup Events** (`dataset2_plastic_waste_event` + `dataset2_waste_collector` -> `cleaned_waste_events.csv`)
3. **Pillar 3: Green Initiatives & Organizations** (`dataset3_green_initiative_project` + `dataset3_organization` -> `cleaned_green_initiatives.csv`)

Each group is processed separately to clean missing values and merge using their respective join keys.

## 1. Setup Environment & Load Datasets

In [1]:
import pandas as pd
import numpy as np
import os

# Define raw and cleaned directories
raw_dir = "../"
cleaned_dir = "../data/"

# Create data directory if it doesn't exist
os.makedirs(cleaned_dir, exist_ok=True)

# Load datasets
df_sample = pd.read_csv(os.path.join(raw_dir, "dataset1_microplastic_sample.csv"))
df_location = pd.read_csv(os.path.join(raw_dir, "dataset1_sampling_location.csv"))
df_event = pd.read_csv(os.path.join(raw_dir, "dataset2_plastic_waste_event.csv"))
df_collector = pd.read_csv(os.path.join(raw_dir, "dataset2_waste_collector.csv"))
df_project = pd.read_csv(os.path.join(raw_dir, "dataset3_green_initiative_project.csv"))
df_org = pd.read_csv(os.path.join(raw_dir, "dataset3_organization.csv"))

print("All raw datasets loaded successfully.")

All raw datasets loaded successfully.


## 2. Process Pillar 1: Microplastic Monitoring

We will merge `df_sample` with `df_location` using the join key `location_id`.

In [2]:
# Check missing values
print("Missing values in df_sample:\n", df_sample.isnull().sum()[df_sample.isnull().sum() > 0])
print("\nMissing values in df_location:\n", df_location.isnull().sum()[df_location.isnull().sum() > 0])

# Inner merge on location_id
df_microplastics = pd.merge(df_sample, df_location, on="location_id", how="inner")

print(f"\nRaw df_sample shape: {df_sample.shape}")
print(f"Raw df_location shape: {df_location.shape}")
print(f"Merged df_microplastics shape (Pillar 1): {df_microplastics.shape}")

# Convert collection_date to datetime
df_microplastics["collection_date"] = pd.to_datetime(df_microplastics["collection_date"])

df_microplastics.head()

Missing values in df_sample:
 Series([], dtype: int64)

Missing values in df_location:
 Series([], dtype: int64)

Raw df_sample shape: (8500, 20)
Raw df_location shape: (8500, 20)
Merged df_microplastics shape (Pillar 1): (8500, 39)


,sample_id,location_id,collector_id,collection_date,plastic_type,shape,color,size_class,size_um,concentration_particles_per_l,...,elevation_m,distance_to_coast_km,area_km2,monitoring_status,established_year,annual_rainfall_mm,avg_temperature_c,pollution_index,biodiversity_score,is_protected_zone
0,71904342,29204927,80336931,2018-12-11,PP,Fragment,Orange,Small(1-5mm),7.76,6.5724,...,2088.1,77.45,13.534,Inactive,1998,1474.5,17.56,9.75,8.59,True
1,47380544,11889398,64517129,2020-02-19,PET,Fragment,Orange,Small(1-5mm),25.19,6.7565,...,3005.0,1013.51,6.718,Active,1995,602.9,40.01,2.81,3.56,True
2,84714128,79961132,35369893,2020-08-31,ABS,Film,Grey,Small(1-5mm),217.69,2.7162,...,1638.4,1804.29,0.913,Inactive,2010,506.4,19.34,6.42,1.11,False
3,95832205,92692727,30883939,2023-11-25,EPS,Film,Red,Small(1-5mm),59.29,32.7071,...,2876.9,90.71,1.956,Active,2017,823.7,19.00,3.01,2.62,False
4,65184018,58999937,84397165,2015-07-08,PP,Fragment,Mixed,Micro(0.001-1mm),161.98,11.8880,...,3382.1,970.64,8.082,Active,2020,619.8,27.61,8.03,5.03,False


## 3. Process Pillar 2: Plastic Waste Cleanup Events

We will merge `df_event` with `df_collector` on `collector_id` after handling missing values in the collector table.

In [3]:
# Check missing values
print("Missing values in df_event:\n", df_event.isnull().sum()[df_event.isnull().sum() > 0])
print("\nMissing values in df_collector before imputation:\n", df_collector.isnull().sum()[df_collector.isnull().sum() > 0])

# Impute missing values in df_collector
# Fill string category 'certification' with 'None'
df_collector["certification"] = df_collector["certification"].fillna("None")

# Fill 'last_audit_year' with -1 representing 'Never Audited'
df_collector["last_audit_year"] = df_collector["last_audit_year"].fillna(-1)

print("\nMissing values in df_collector after imputation:\n", df_collector.isnull().sum()[df_collector.isnull().sum() > 0])

# Inner merge on collector_id
df_waste_events = pd.merge(df_event, df_collector, on="collector_id", how="inner")

print(f"\nRaw df_event shape: {df_event.shape}")
print(f"Raw df_collector shape: {df_collector.shape}")
print(f"Merged df_waste_events shape (Pillar 2): {df_waste_events.shape}")

# Convert event_date to datetime
df_waste_events["event_date"] = pd.to_datetime(df_waste_events["event_date"])

df_waste_events.head()

Missing values in df_event:
 Series([], dtype: int64)

Missing values in df_collector before imputation:
 certification      3007
last_audit_year    1205
dtype: int64

Missing values in df_collector after imputation:
 Series([], dtype: int64)

Raw df_event shape: (8500, 20)
Raw df_collector shape: (8500, 20)
Merged df_waste_events shape (Pillar 2): (8500, 39)


,event_id,collector_id,site_id,event_date,waste_source,plastic_grade,weight_collected_ton,volume_collected_m3,microplastic_fraction_pct,disposal_mode,...,certification,funding_source,avg_cost_per_ton_usd,recycling_rate_pct,gps_tracking_enabled,partnership_count,carbon_offset_ton_yr,social_impact_score,is_active,last_audit_year
0,43136592,71693560,34744332,2023-10-20,Agriculture,Recycled Grade A,72.6568,64.8841,36.11,Ocean Discharge,...,GreenStar,Government Grant,1154.39,37.93,True,4,2818.53,2.62,True,2022.0
1,22324466,10406314,62605800,2024-10-22,Electronics,Mixed,0.3230,0.2435,15.30,Upcycled,...,LEED,Corporate Sponsorship,820.26,96.76,True,6,1109.52,9.88,True,2022.0
2,20044529,54609134,34256785,2023-08-23,Fishing Industry,Virgin,0.1452,0.2902,33.37,Landfill,...,ISO14001,Community Donation,969.80,62.39,True,73,1589.22,1.80,True,2023.0
3,98583108,33656261,14169480,2018-07-20,Fishing Industry,Recycled Grade A,15.9216,52.0416,0.69,Landfill,...,ISO14001,Government Grant,784.61,80.42,True,31,4278.01,4.99,True,2024.0
4,50957490,34426567,75285098,2019-12-10,Packaging,Virgin,1.5105,4.4977,3.05,Landfill,...,ISO9001,Government Grant,726.85,81.83,True,75,504.51,8.88,False,-1.0


## 4. Process Pillar 3: Green Initiatives & Organizations

We will merge `df_project` with `df_org` on `organization_id` after handling missing values in both tables.

In [4]:
# Check missing values
print("Missing values in df_project before imputation:\n", df_project.isnull().sum()[df_project.isnull().sum() > 0])
print("\nMissing values in df_org before imputation:\n", df_org.isnull().sum()[df_org.isnull().sum() > 0])

# Impute missing values in df_project
df_project["partner_sector"] = df_project["partner_sector"].fillna("None")
df_project["policy_outcome"] = df_project["policy_outcome"].fillna("None")

# Impute missing values in df_org
df_org["last_report_year"] = df_org["last_report_year"].fillna(-1)

print("\nMissing values in df_project after imputation:\n", df_project.isnull().sum()[df_project.isnull().sum() > 0])
print("\nMissing values in df_org after imputation:\n", df_org.isnull().sum()[df_org.isnull().sum() > 0])

# Inner merge on organization_id
df_green_initiatives = pd.merge(df_project, df_org, on="organization_id", how="inner")

print(f"\nRaw df_project shape: {df_project.shape}")
print(f"Raw df_org shape: {df_org.shape}")
print(f"Merged df_green_initiatives shape (Pillar 3): {df_green_initiatives.shape}")

# Convert dates to datetime
df_green_initiatives["start_date"] = pd.to_datetime(df_green_initiatives["start_date"])
df_green_initiatives["end_date"] = pd.to_datetime(df_green_initiatives["end_date"])

df_green_initiatives.head()

Missing values in df_project before imputation:
 partner_sector     594
policy_outcome    2127
dtype: int64

Missing values in df_org before imputation:
 last_report_year    1379
dtype: int64

Missing values in df_project after imputation:
 Series([], dtype: int64)

Missing values in df_org after imputation:
 Series([], dtype: int64)

Raw df_project shape: (8500, 20)
Raw df_org shape: (8500, 20)
Merged df_green_initiatives shape (Pillar 3): (8500, 39)


,project_id,organization_id,lead_collector_id,project_name,initiative_type,target_plastic_type,geographic_scope,start_date,end_date,project_status,...,num_projects_active,num_countries_operating,has_un_consultative_status,transparency_rating,media_presence_score,co2_reduction_target_ton_yr,plastic_reduction_target_ton_yr,policy_influence_score,is_accredited,last_report_year
0,66937928,37014776,86827451,Initiative_37928,Tech Innovation,All Types,Ocean Basin,2022-05-14,2024-04-03,Completed,...,51,34,False,2.63,68.74,18412.0,68927.0,5.65,True,2022.0
1,50799628,93061673,54350141,Initiative_99628,Policy Lobbying,Agricultural Film,National,2014-01-11,2017-03-08,Completed,...,129,49,False,4.52,80.16,429056.0,99654.0,2.61,True,2021.0
2,29523847,67987946,74954318,Initiative_23847,Policy Lobbying,Synthetic Textiles,National,2021-01-31,2023-04-02,Completed,...,189,57,False,4.25,89.11,650246.0,497168.0,2.04,True,2023.0
3,60601294,10734040,49716662,Initiative_01294,Cleanup Campaign,Single-Use Plastics,Global,2013-12-15,2018-12-10,Active,...,249,71,True,4.09,61.25,873561.0,348591.0,2.26,False,2024.0
4,31170152,29243518,54421508,Initiative_70152,Recycling Infrastructure,Packaging,National,2020-09-07,2021-01-14,Planning,...,137,35,False,1.92,79.72,341360.0,150646.0,0.21,False,2022.0


## 5. Export Cleaned & Merged Datasets

Save the integrated dataframes into the `/data/` folder for exploration and modeling.

In [5]:
# Export to CSV
path_micro = os.path.join(cleaned_dir, "cleaned_microplastics.csv")
path_waste = os.path.join(cleaned_dir, "cleaned_waste_events.csv")
path_green = os.path.join(cleaned_dir, "cleaned_green_initiatives.csv")

df_microplastics.to_csv(path_micro, index=False)
df_waste_events.to_csv(path_waste, index=False)
df_green_initiatives.to_csv(path_green, index=False)

print("Three integrated datasets exported successfully:")
print(f"1. {path_micro} (shape: {df_microplastics.shape})")
print(f"2. {path_waste} (shape: {df_waste_events.shape})")
print(f"3. {path_green} (shape: {df_green_initiatives.shape})")

Three integrated datasets exported successfully:
1. ../data/cleaned_microplastics.csv (shape: (8500, 39))
2. ../data/cleaned_waste_events.csv (shape: (8500, 39))
3. ../data/cleaned_green_initiatives.csv (shape: (8500, 39))
